In [8]:
import numpy as np
import pandas as pd

#  Charger les données

# Charger le dataset
df = pd.read_csv("data.csv")

# Supprimer les colonnes inutiles
if 'Unnamed: 32' in df.columns:
    df = df.drop(columns=['Unnamed: 32'])

if 'id' in df.columns:
    df = df.drop(columns=['id'])

# Transformer la variable cible :
# M (Malignant) → 1
# B (Benign) → 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

# Séparer les variables explicatives (X) et la cible (y)
X = df.drop(columns=['diagnosis']).values.astype(float)
y = df['diagnosis'].values.astype(float)


# Train / Test split

# Mélanger les indices pour éviter biais
np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

# 80% entraînement / 20% test
split = int(0.8 * len(X))

X_train = X[indices[:split]]
y_train = y[indices[:split]]

X_test = X[indices[split:]]
y_test = y[indices[split:]]




In [9]:
#  Arbre simple 
# Un stump = arbre très simple avec 1 seule règle
class DecisionStump:
    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Initialisation
        self.feature = None
        self.threshold = None
        self.left_value = None
        self.right_value = None

        best_error = float('inf')  # erreur minimale

        # Tester toutes les variables
        for feature in range(n_features):

            # Tester tous les seuils possibles
            thresholds = np.unique(X[:, feature])

            for t in thresholds:

                # Diviser les données en deux groupes
                left = X[:, feature] <= t
                right = X[:, feature] > t

                # Ignorer si division invalide
                if np.sum(left) == 0 or np.sum(right) == 0:
                    continue

                # Calculer la moyenne des valeurs (régression)
                left_value = np.mean(y[left])
                right_value = np.mean(y[right])

                # Créer les prédictions
                y_pred = np.zeros(n_samples)
                y_pred[left] = left_value
                y_pred[right] = right_value

                # Calcul de l’erreur (MSE)
                error = np.mean((y - y_pred) ** 2)

                # Garder la meilleure séparation
                if error < best_error:
                    best_error = error
                    self.feature = feature
                    self.threshold = t
                    self.left_value = left_value
                    self.right_value = right_value

    def predict(self, X):
        y_pred = np.zeros(X.shape[0])

        # Appliquer la règle trouvée
        left = X[:, self.feature] <= self.threshold
        right = X[:, self.feature] > self.threshold

        y_pred[left] = self.left_value
        y_pred[right] = self.right_value

        return y_pred


#  XGBoost simplifié (Gradient Boosting)

class SimpleXGBoost:
    def __init__(self, n_estimators=20, learning_rate=0.1):
        # n_estimators : nombre d’arbres
        # learning_rate : vitesse d’apprentissage
        self.n_estimators = n_estimators
        self.lr = learning_rate
        self.models = []

    def fit(self, X, y):

        # Prédiction initiale = moyenne des y
        self.base_pred = np.mean(y)

        # Initialiser les prédictions
        y_pred = np.full(len(y), self.base_pred)

        # Boucle de boosting
        for _ in range(self.n_estimators):

            # Calcul des erreurs (résidus)
            residuals = y - y_pred

            # Entraîner un arbre sur les erreurs
            stump = DecisionStump()
            stump.fit(X, residuals)

            # Prédire la correction
            update = stump.predict(X)

            # Mise à jour des prédictions
            y_pred += self.lr * update

            # Sauvegarder le modèle
            self.models.append(stump)

    def predict(self, X):

        # Commencer avec la prédiction de base
        y_pred = np.full(X.shape[0], self.base_pred)

        # Ajouter les corrections de chaque arbre
        for model in self.models:
            y_pred += self.lr * model.predict(X)

        # Transformer en classe (0 ou 1)
        return (y_pred > 0.5).astype(int)




In [10]:
#  Entraîner le modèle

model = SimpleXGBoost(n_estimators=20, learning_rate=0.1)
model.fit(X_train, y_train)


#  Prédictions

y_pred = model.predict(X_test)


# Évaluation simple

correct = 0
incorrect = 0

for i in range(len(y_test)):
    if y_test[i] == y_pred[i]:
        correct += 1
    else:
        incorrect += 1

print("Évaluation XGBoost (simplifié) :")
print("Bonnes prédictions :", correct)
print("Mauvaises prédictions :", incorrect)
print("Total :", len(y_test))

Évaluation XGBoost (simplifié) :
Bonnes prédictions : 102
Mauvaises prédictions : 12
Total : 114
